In [0]:
silver_txn = spark.table(
    "banking_catalog.banking_schema.silver_transactions"
)

silver_acc = spark.table(
    "banking_catalog.banking_schema.silver_accounts"
)

silver_txn.select("from_bank").distinct().show(10, False)

silver_acc.select("bank_id").distinct().show(10, False)

+---------+
|from_bank|
+---------+
|0343742  |
|011056   |
|0022422  |
|0327675  |
|0152416  |
|0243807  |
|0013078  |
|0134091  |
|0117698  |
|0046616  |
+---------+
only showing top 10 rows
+-------+
|bank_id|
+-------+
|331579 |
|210    |
|21884  |
|32742  |
|127390 |
|224555 |
|32013  |
|335355 |
|1132   |
|217824 |
+-------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import regexp_replace

txn_banks = (
    silver_txn
    .select(
        regexp_replace("from_bank", "^0+", "").alias("bank_id")
    )
    .distinct()
)

acc_banks = (
    silver_acc
    .select("bank_id")
    .distinct()
)

txn_banks.intersect(acc_banks).count()

30470

In [0]:
txn_banks.count()
acc_banks.count()

30470

In [0]:
txn_banks.intersect(acc_banks).count()

30470

In [0]:
from pyspark.sql.functions import *

silver_txn = spark.table(
    "banking_catalog.banking_schema.silver_transactions"
)

silver_acc = spark.table(
    "banking_catalog.banking_schema.silver_accounts"
)

bank_summary_df = (
    silver_txn.alias("t")
    .join(
        silver_acc.alias("a"),
        regexp_replace(col("t.from_bank"), "^0+", "")
        == col("a.bank_id"),
        "inner"
    )
)

In [0]:
gold_bank_df = (
    bank_summary_df
    .groupBy(
        col("a.bank_id"),
        col("a.bank_name")
    )
    .agg(
        count("*").alias("transaction_count"),
        sum("amount_paid").alias("total_amount"),
        sum(
            when(col("is_laundering") == 1, 1)
            .otherwise(0)
        ).alias("laundering_transaction_count")
    )
)

In [0]:
gold_bank_df.count()

30470

In [0]:
display(
    gold_bank_df.orderBy(
        col("total_amount").desc()
    )
)

bank_id,bank_name,transaction_count,total_amount,laundering_transaction_count
15,Japan Bank #0,160125633,1673528368632135.12,140346
12,National Bank of the East,252102394,1108292540540391.67,240236
22164,Spruce Cooperative Bank,5244456,889305971157199.32,4712
7,India Bank #25,51758190,793295709802664.40,18315
220,National Bank of Pittsburgh,132523930,685718473216650.50,55660
112064,India Bank #26,5970160,652443283160468.80,1120
12381,Bank of the East,5698218,634524766854074.76,7272
10,National Bank of Laramie,253702932,600661429518234.00,158508
19925,Japan Bank #36,2131920,485295610344557.76,2160
3,China Bank #14,68259901,458850198110529.87,23101


In [0]:
gold_bank_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.banking_schema.gold_bank_summary"
    )

In [0]:
gold_bank_df.count() 

30470